<a href="https://colab.research.google.com/github/GuilhermeFlorencio32/GEOAI-PARA-FACHADAS-V1/blob/main/GEOAI_REVISAO" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#estabelecendo as variáveis;
print('xpix = xpix do centro do objeto'
'xmax = valor máximo da dimensão de x no panorama'
'alpha = yaw = da intersecção das fotos 0° a ao norte do panorama'
'beta = distância da interseccção até a direção do objeto'
'gama = azimute, que será usado para ponderar a direção e atribuir o objeto')


In [ ]:
import math

In [ ]:
'estabelecendo as relações'
import math
import requests
import base64
from PIL import Image
import io

# === CONFIGURAÇÕES ===
API_KEY = "00ivd23tUucrDPOWwuTd" # Chave de API do roboflow
PROJECT_ID = "modelo-yolo-v12-v11-cajazeiras" # ID do modelo
VERSION = 7 # versionamento do modelo
ALPHA = 330.977336  # yaw da câmera (norte do panorama)

# Imagem usada para teste
image_path = "/content/20250809_113719_008895.jpg"

# === CARREGAR IMAGEM ===
image = Image.open(image_path)
xmax = image.width  # largura total da panorama

# Converter para base64
buffered = io.BytesIO()
image.save(buffered, format="JPEG")
img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

# === ENVIAR PARA API ROBOFLOW === Realiza a modelagem da imagem do panorama

response = requests.post(
    f"https://detect.roboflow.com/{PROJECT_ID}/{VERSION}",
    params={"api_key": API_KEY},
    data=img_base64,
    headers={"Content-Type": "application/x-www-form-urlencoded"}
)

predictions = response.json()["predictions"] # Puxa os atributos da modelagem dos panoramas ex: confiança, xpix, afins


import math
import geopandas as gpd
from shapely.geometry import LineString, Point

HEADING = 330.977336          # do metadado
AZIMUTE_PIXEL0 = (HEADING - 180) % 360  # = 150.977336° - Visada do norte do panorama para a fachada

#Coordenadas do panorama
E_origem = 565095.293346
N_origem = 8574606.399346
distancia_m = 50

linhas = [] # Guarda a geometria das linhas
atributos = [] # Vai guardar, para cada linha correspondente, um dicionário com os metadados daquela detecção: xpix, confiança,az

for pred in predictions:
    xpix = pred["x"]
    classe = pred["class"]
    confianca = pred["confidence"]

# Cálculo do azimute de cada visada
    beta = xpix * 360 / xmax
    gama = (AZIMUTE_PIXEL0 + beta) % 360   # azimute geográfico correto

# Defina a direção das linhas com base no azimute
    gama_rad = math.radians(gama)
    E_final = E_origem + distancia_m * math.sin(gama_rad)
    N_final = N_origem + distancia_m * math.cos(gama_rad)

# Criando linha e atributos da linha
    linha = LineString([(E_origem, N_origem), (E_final, N_final)])
    linhas.append(linha)
    atributos.append({
        "classe": classe,
        "confianca": round(confianca, 2),
        "xpix": round(xpix, 1),
        "beta": round(beta, 2),
        "azimute": round(gama, 2)
    })
#print das informações de cada visada
    print(f"Classe: {classe} | xpix: {xpix:.1f} | beta: {beta:.2f}° | azimute: {gama:.2f}°")
# gerando geopackage com as linhas das visadas(50m)
gdf = gpd.GeoDataFrame(atributos, geometry=linhas, crs="EPSG:31984")
gdf.to_file("azimutes_fachadas.gpkg", driver="GPKG")

print("Arquivo salvo!")

In [ ]:
# =============================================================================
# INTERSEÇÃO LINHAS x LOTES (GeoPackage) - Google Colab
#
# Como usar:
#   - Ter em mãos o arquivo de linhas em gpkg que foi gerado
# =============================================================================

# %% 1) Instalação das dependências

!pip install geopandas -q
!pip install geopandas pyogrio -q

# %% 2) Upload dos arquivos GeoPackage

from google.colab import files
import os

print("Selecione o arquivo .gpkg de LINHAS:")
upload_linhas = files.upload()
CAMINHO_LINHAS = list(upload_linhas.keys())[0]

#Importando Camada Lotes Via Drive - fixa e mais rápida
from google.colab import drive
print(" O arquivo .gpkg de LOTES está sendo carregado")
drive.mount('/content/drive')
CAMINHO_LOTES = "/content/drive/MyDrive/Camada Lotes - APLICAÇÃO GEOAI/camada lote geopackage.gpkg"

print("Arquivo de linhas:", CAMINHO_LINHAS)
print("Arquivo de lotes:", CAMINHO_LOTES)

# %% 3) Configurações
# Um .gpkg pode conter mais de uma camada dentro do mesmo arquivo.
# Se for o seu caso, informe o NOME EXATO da camada aqui. Deixe None se
# o arquivo tiver apenas 1 camada (o script detecta e usa automaticamente).
NOME_CAMADA_LINHAS = None  # ex: "linhas_projeto"
NOME_CAMADA_LOTES = None   # ex: "lotes_urbanos"

# Campo de ID já existente (opcional). Deixe None para o script criar um automático.
CAMPO_ID_LINHA = None  # ex: "id"
CAMPO_ID_LOTE = "id"   # ex: "id"

# %% 4) Leitura dos GeoPackages
import geopandas as gpd
import pandas as pd

# Retorna a camda que foi usada, possível retirada do código
def listar_camadas(caminho):
    """Retorna a lista de camadas contidas em um .gpkg, com fallback caso
    a versão do geopandas instalada não tenha gpd.list_layers."""
    try:
        return gpd.list_layers(caminho)["name"].tolist()
    except AttributeError:
        import fiona
        return fiona.listlayers(caminho)


def ler_gpkg(caminho, nome_camada=None):
    camadas = listar_camadas(caminho)
    if nome_camada is None:
        if len(camadas) > 1:
            print(
                f"AVISO: '{caminho}' contém {len(camadas)} camadas {camadas}. "
                f"Usando a primeira: '{camadas[0]}'."
            )
            print(
                "Se não for a camada desejada, defina NOME_CAMADA_LINHAS/"
                "NOME_CAMADA_LOTES na célula anterior e rode de novo."
            )
        nome_camada = camadas[0]
    return gpd.read_file(caminho, layer=nome_camada)


gdf_linhas = ler_gpkg(CAMINHO_LINHAS, NOME_CAMADA_LINHAS)
gdf_lotes = ler_gpkg(CAMINHO_LOTES, NOME_CAMADA_LOTES)

print(f"Linhas carregadas: {len(gdf_linhas)}")
print(f"Lotes carregados: {len(gdf_lotes)}")

# --- INÍCIO DA CORREÇÃO ---
# %% 5) Preparação e Interseção espacial
# Se CAMPO_ID_LINHA for None, criar um ID temporário para as linhas
if CAMPO_ID_LINHA is None:
    gdf_linhas['_temp_id_linha'] = gdf_linhas.index
    CAMPO_ID_LINHA = '_temp_id_linha'

# Se CAMPO_ID_LOTE for None, criar um ID temporário para os lotes (embora já esteja 'id' por padrão)
if CAMPO_ID_LOTE is None:
    gdf_lotes['_temp_id_lote'] = gdf_lotes.index
    CAMPO_ID_LOTE = '_temp_id_lote'

# Garante que os CRS das camadas são os mesmos para o sjoin
# Se forem diferentes, transforma gdf_lotes para o CRS de gdf_linhas
# Pode ser removido para fins de otimização, deixar print alertando
if gdf_linhas.crs != gdf_lotes.crs:
    print(f"AVISO: CRS de gdf_linhas ({gdf_linhas.crs}) e gdf_lotes ({gdf_lotes.crs}) são diferentes. Transformando gdf_lotes para o CRS de gdf_linhas.")
    gdf_lotes = gdf_lotes.to_crs(gdf_linhas.crs)

print("Realizando intersecção entre as linhas e os lotes...")
# Realiza um join espacial para encontrar os lotes que cada linha intercepta
join = gpd.sjoin(gdf_linhas, gdf_lotes, how="inner", predicate="intersects")

print(f"Número de lotes intersectados: {len(join)}")
# --- FIM DA CORREÇÃO ---

# %% 6) Vértices de interseção linha x borda do lote, distância e associação
from shapely.geometry import Point

# Acesso rápido às geometrias originais (linhas e lotes) via id
geom_linhas_dict = gdf_linhas.set_index(CAMPO_ID_LINHA)['geometry'].to_dict()
geom_lotes_dict = gdf_lotes.set_index(CAMPO_ID_LOTE)['geometry'].to_dict()

registros_vertices = []   # armazena a camada de vértices e suas atribuições, ex: geom,posição
registros_distancia = []  # id_linha, id_lote, distancia_min -> usado pra escolher o vencedor , tirando a distância

for _, row in join.iterrows():
    id_linha = row[CAMPO_ID_LINHA]
    id_lote = row[CAMPO_ID_LOTE]
    geom_linha = geom_linhas_dict[id_linha]
    geom_lote = geom_lotes_dict[id_lote]

    # Ponto inicial da linha (referência fixa para medir distância)
    ponto_inicial = Point(geom_linha.coords[0])

    # Interseção da linha com o CONTORNO (borda) do lote
    intersecao = geom_linha.intersection(geom_lote.boundary)

    pontos = []
    if intersecao.is_empty:
        pontos = []
    elif intersecao.geom_type == "Point":
        pontos = [intersecao]
    elif intersecao.geom_type == "MultiPoint":
        pontos = list(intersecao.geoms)
    else:
        # GeometryCollection (caso raro de sobreposição linha/borda) -> filtra só pontos
        for geom in getattr(intersecao, "geoms", [intersecao]):
            if geom.geom_type == "Point":
                pontos.append(geom)

    if len(pontos) == 0:
        # Linha totalmente dentro do lote (não cruza a borda) -> vence automaticamente
        distancia_min = 0.0
    else:
        # Distância do PONTO INICIAL da linha até cada vértice de cruzamento
        distancias = [ponto_inicial.distance(p) for p in pontos]
        distancia_min = min(distancias)
        for p, d in zip(pontos, distancias):
            registros_vertices.append({
                CAMPO_ID_LINHA: id_linha,
                CAMPO_ID_LOTE: id_lote,
                "distancia": d,
                "geometry": p
            })

    registros_distancia.append({
        CAMPO_ID_LINHA: id_linha,
        CAMPO_ID_LOTE: id_lote,
        "distancia_min": distancia_min
    })

# amada de vértices (pontos de interseção linha x borda do lote)
gdf_vertices = gpd.GeoDataFrame(registros_vertices, geometry="geometry", crs=gdf_linhas.crs)
print(f"Vértices de interseção gerados: {len(gdf_vertices)}")

# Escolhe, para cada linha, o lote com a MENOR distância (vencedor)
df_distancias = pd.DataFrame(registros_distancia)
idx_vencedor = df_distancias.groupby(CAMPO_ID_LINHA)["distancia_min"].idxmin()
lote_vencedor = df_distancias.loc[idx_vencedor, [CAMPO_ID_LINHA, CAMPO_ID_LOTE, "distancia_min"]].rename(
    columns={CAMPO_ID_LOTE: "lote_associado", "distancia_min": "distancia_lote_associado"}
)

# Resumo por linha (lista de todos os lotes interceptados, como antes)
resumo_linha = (
    join.groupby(CAMPO_ID_LINHA)[CAMPO_ID_LOTE]
    .agg(list)
    .reset_index()
    .rename(columns={CAMPO_ID_LOTE: "lotes_interceptados"})
)
resumo_linha["qtd_lotes"] = resumo_linha["lotes_interceptados"].apply(len)
resumo_linha["flag_multiplos_lotes"] = resumo_linha["qtd_lotes"] > 1

# Junta o vencedor (lote_associado + distância) ao resumo
resumo_linha = resumo_linha.merge(lote_vencedor, on=CAMPO_ID_LINHA, how="left")

gdf_linhas_resultado = gdf_linhas.merge(resumo_linha, on=CAMPO_ID_LINHA, how="left")
gdf_linhas_resultado["lotes_interceptados"] = gdf_linhas_resultado["lotes_interceptados"].apply(
    lambda x: ",".join(map(str, x)) if isinstance(x, list) else ""
)
gdf_linhas_resultado["qtd_lotes"] = gdf_linhas_resultado["qtd_lotes"].fillna(0).astype(int)
gdf_linhas_resultado["flag_multiplos_lotes"] = gdf_linhas_resultado["flag_multiplos_lotes"].fillna(False)
# %% 7) Resultado por LOTE -> quantas linhas o interceptam + sinalização

resumo_lote = (
    join.groupby(CAMPO_ID_LOTE)[CAMPO_ID_LINHA]
    .agg(list)
    .reset_index()
    .rename(columns={CAMPO_ID_LINHA: "linhas_associadas"})
)
resumo_lote["qtd_linhas"] = resumo_lote["linhas_associadas"].apply(len)
resumo_lote["flag_multiplas_linhas"] = resumo_lote["qtd_linhas"] > 1  # lote tocado por >1 linha

gdf_lotes_resultado = gdf_lotes.merge(resumo_lote, on=CAMPO_ID_LOTE, how="inner")
gdf_lotes_resultado["linhas_associadas"] = gdf_lotes_resultado["linhas_associadas"].apply(
    lambda x: ",".join(map(str, x)) if isinstance(x, list) else ""
)

# %% 7.1) Planilha exclusiva: apenas lotes interceptados por MAIS DE 1 linha
gdf_lotes_multiplas_linhas = gdf_lotes_resultado[
    gdf_lotes_resultado["flag_multiplas_linhas"] == True
].copy()

print(f"Lotes com mais de 1 linha associada: {len(gdf_lotes_multiplas_linhas)}")

# %% 8) Resumo em tela
qtd_lotes_flag = gdf_lotes_resultado["flag_multiplas_linhas"].sum()
qtd_linhas_flag = gdf_linhas_resultado["flag_multiplos_lotes"].sum()
print(f"Lotes interceptados por mais de 1 linha: {qtd_lotes_flag}")
print(f"Linhas que interceptam mais de 1 lote:   {qtd_linhas_flag}")

# %% 9) Exportação dos resultados (GeoPackage + CSV com o resumo)
# GeoPackage não tem o limite de 10 caracteres do .dbf do shapefile, então os
# nomes de coluna acima já ficam completos e legíveis no arquivo de saída.
os.makedirs("saida", exist_ok=True)

CAMINHO_SAIDA_GPKG = "saida/resultado_intersecao.gpkg"
# As duas camadas de resultado ficam dentro do MESMO arquivo .gpkg
gdf_linhas_resultado.to_file(CAMINHO_SAIDA_GPKG, layer="linhas_resultado", driver="GPKG")
gdf_lotes_resultado.to_file(CAMINHO_SAIDA_GPKG, layer="lotes_resultado", driver="GPKG")
# Camada extra dentro do mesmo .gpkg, só com os lotes "conflituosos"
gdf_lotes_multiplas_linhas.to_file(
    CAMINHO_SAIDA_GPKG, layer="lotes_multiplas_linhas", driver="GPKG"
)
# Camada extra: vértices de interseção
gdf_vertices.to_file(CAMINHO_SAIDA_GPKG, layer="vertices_intersecao", driver="GPKG")

# CSVs de apoio (fáceis de abrir em planilha, sem precisar de SIG)
gdf_linhas_resultado.drop(columns="geometry").to_csv("saida/linhas_resultado.csv", index=False)
gdf_lotes_resultado.drop(columns="geometry").to_csv("saida/lotes_resultado.csv", index=False)
# CSV exclusivo desses lotes
gdf_lotes_multiplas_linhas.drop(columns="geometry").to_csv("saida/lotes_multiplas_linhas.csv", index=False)
# CSV dos vértices
gdf_vertices.drop(columns="geometry").to_csv("saida/vertices_intersecao.csv", index=False)
# Compacta a pasta de saída em um único .zip para download
import shutil
shutil.make_archive("resultado_intersecao", "zip", "saida")

# %% 10) Download do resultado
files.download("resultado_intersecao.zip")